# X6Y3 tune-up with LeeQ built-in experiments

Same usage as [TuneUpExample.ipynb](TuneUpExample.ipynb): initialize the standard LeeQ DUT, then run one experiment per cell. LeeQ handles pulse collections, acquisition, fitting, plots and calibration updates.

**Experiment cells take shots when executed.** Run cells individually and inspect each result. The calibration work remains paused until you run them. Use the **LeeQ — Huracan experiments** kernel. After replacing the previous notebook, restart the kernel and reload this file.

The standard `SimpleDriveCollection` owns X/Xp/Xm. It is initialized from the supplied notebook's X-pulse parameters; there is no custom X90 adapter. `update=True` changes the in-memory DUT. The original X6Y3 YAML/ZIP are not overwritten.

## Load LeeQ and the Huracan setup

In [ ]:
import os, sys
from pathlib import Path

LEEQ_ROOT = Path('/local/data/projects/LeeQ-lbnl-qubic')
HARNESS_ROOT = Path('/local/data/projects/qubic_validation')
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LAB_CHRONICLE_LOG_DIR'] = str(HARNESS_ROOT / 'artifacts/x6y3/leeq-chronicle')
sys.path.insert(0, str(LEEQ_ROOT))
sys.path.append(str(HARNESS_ROOT / 'artifacts/x6y3/venv/lib/python3.11/site-packages'))

from leeq.chronicle import Chronicle
from leeq.core.primitives.built_in.common import Delay
from leeq.utils.compatibility import prims
# This checkout's built-ins use the legacy Delay export.
prims.Delay = Delay
from leeq.experiments.experiments import ExperimentManager
from leeq.core.elements.built_in.qudit_transmon import TransmonElement
from leeq.core.primitives.built_in.simple_drive import SimpleDrive
from leeq.setups.huracan import create_huracan_setup
from leeq.setups.x6y3 import X6Y3Calibration
from leeq.experiments.builtin import (
    MeasurementCalibrationMultilevelGMM,
    SimpleRamseyMultilevel,
    AmpPingpongCalibrationSingleQubitMultilevel,
    SimpleT1,
)

manager = ExperimentManager()
if 'Huracan-X6Y3' in manager.get_available_setup_names():
    huracan = manager.get_setup('Huracan-X6Y3')
else:
    huracan = create_huracan_setup('/local/data/artifacts/zcu216/20260903T_aa01c78f_rdrv230_d4_candidate/bits_aa01c78f/channel_config.json')
    manager.register_setup(huracan)
manager.set_default_setup(huracan.name)

calibration = X6Y3Calibration(HARNESS_ROOT / 'X6Y3/configs_new/config.yaml')
calibration.verify_channel_metadata(huracan.channel_metadata)
huracan.status.set_parameters(Engine_Batch_Size=1, Shot_Number=1024,
                              Shot_Period=calibration.passive_delay_us,
                              Acquisition_Type='IQ', Measurement_Basis=None,
                              Plot_Result_In_Jupyter=True, AIAutoInspectPlots=False)
if not Chronicle().is_recording():
    Chronicle().start_log('X6Y3-builtins')

## Initialize standard LeeQ qubits

The existing X6Y3 importer supplies the physical X-drive and readout parameters. The DUT and drive collection below are LeeQ's standard classes.

In [ ]:
duts = {}
for q in (0, 1):
    drive = next(p for p in calibration.gate(q, 'X').children if isinstance(p, SimpleDrive))
    drive_params = dict(drive.get_parameters(), type='SimpleDriveCollection')
    readout_params = dict(calibration.measurement(q).get_parameters(), type='SimpleDispersiveMeasurement')
    duts[f'Q{q}'] = TransmonElement(name=f'Q{q}', parameters={
        'lpb_collections': {'f01': drive_params},
        'measurement_primitives': {'0': readout_params},
    })

In [ ]:
dut = duts['Q0']     # choose Q0 or Q1
c1 = dut.get_c1('f01')
mprim = dut.get_measurement_prim_intlist(0)

## Readout calibration

Run this first to install LeeQ's GMM measurement transform for the subsequent experiments.

In [ ]:
calib = MeasurementCalibrationMultilevelGMM(dut=dut, mprim_index=0, sweep_lpb_list=(c1['I'], c1['X']))

## Ramsey — 10 MHz offset, 0.3 µs window

In [ ]:
ramsey_coarse = SimpleRamseyMultilevel(dut=dut, set_offset=10, stop=0.3, step=0.005, update=True)

## Ramsey — 1 MHz offset, 3 µs window

In [ ]:
ramsey_medium = SimpleRamseyMultilevel(dut=dut, set_offset=1, stop=3, step=0.05, update=True)

## Ramsey — 0.1 MHz offset, 30 µs window

Inspect the preceding fit before running this cell. The earlier Q0 fine scan was unreliable; do not assume a long-window fit is valid solely because it returned a number.

In [ ]:
ramsey_fine = SimpleRamseyMultilevel(dut=dut, set_offset=0.1, stop=30, step=0.5, update=True)

## Amplitude fine tuning with ping-pong

`AmpPingpongCalibrationSingleQubitMultilevel` is the current built-in amplitude tuner corresponding to the example's older `AmpTuneUpSingleQubitMultilevel` name. LeeQ updates the standard drive collection and handles X/X90. Start with three iterations; adjust the call if needed.

In [ ]:
pingpong = AmpPingpongCalibrationSingleQubitMultilevel(dut=dut, iteration=3, points=10)

## T1

In [ ]:
t1 = SimpleT1(qubit=dut, time_length=350, time_resolution=7)

## Inspect the current parameters

Experiments display their own plots and record through LeeQ Chronicle. Inspect their result objects (`ramsey_coarse`, `ramsey_medium`, `ramsey_fine`, `pingpong`, `t1`) as needed.

In [ ]:
dut.print_config_info()

## Save reviewed parameters to the LeeQ calibration log

Optional: this saves a local LeeQ calibration log, leaving the original YAML/ZIP intact.

In [ ]:
dut.save_calibration_log()